# Residual Stream Analysis

TODO

## Setup

In [ ]:
from __future__ import annotations

%load_ext autoreload
%autoreload 2

In [ ]:
import torch
from ipywidgets import widgets

from latent_dynamics.dayang.activations import Activations, extract_activations
from latent_dynamics.dayang.analysis import analyze_per_layer, analyze_per_token
from latent_dynamics.dayang.data import DATASET_REGISTRY, load_dataset_from_spec
from latent_dynamics.dayang.model import (
    MODEL_REGISTRY,
    get_token_embeddings,
    get_token_groups,
    load_model_and_tokenizer,
    print_token_groups,
)
from latent_dynamics.dayang.utils import Cache, select
from latent_dynamics.dayang.visualizations import (
    plot_reader_statistics,
    plot_per_layer,
    plot_per_token,
    plot_token_embeddings,
)
from latent_dynamics.dayang.widgets import (
    ActivationsExtractorWidget,
    ActivationsSelectorWidget,
    TokenEmbeddingsLoaderWidget,
    ReadersSelectorWidget,
    ShowHideCheckbox,
    TopKExtractorWidget,
)

torch.set_grad_enabled(False)

print(f"Available models:{'\n - '.join([''] + list(MODEL_REGISTRY))}")
print(f"Available datasets:{'\n - '.join([''] + list(DATASET_REGISTRY))}")

## Interactive analysis

In [ ]:
def analyze_interactive():
    CACHE = Cache(maxsize=10, encoders={Activations: lambda x: (sorted(x.samples.index.tolist()), sorted(x.layers))})

    # Define layer 1 widgets for extracting activations
    out_layer1 = widgets.Output()
    w_act_extractor = ActivationsExtractorWidget(
        out=out_layer1,
        search_path="../activations",
        model="gemma3_1b",
        dataset="xstest",
    )
    w_topk_extractor = TopKExtractorWidget(out=out_layer1, model="gemma3_1b")
    w_emb_loader = TokenEmbeddingsLoaderWidget(out=out_layer1, model="gemma3_1b")
    box_adv_options = widgets.VBox([w_topk_extractor, w_emb_loader])
    box_layer1 = widgets.VBox(
        [
            widgets.HTML("<big><b>Extract activations</b></big>"),
            w_act_extractor,
            ShowHideCheckbox(box_adv_options, value=False, description="Show advanced options", indent=False),
            box_adv_options,
            out_layer1,
        ]
    )

    # Define layer 2 widgets for analyzing activations
    w_analyze_selector = widgets.Dropdown(options=["per layer", "per token"], value="per layer", description="Analyze")
    w_readers_selector = ReadersSelectorWidget()
    w_train_separate = widgets.Checkbox(value=True, description="Analyze separately")
    w_eval_separate = widgets.Checkbox(value=True, description="Visualize separately")
    w_eval_link = widgets.Checkbox(value=True, description="Same as analysis")

    w_train_selector = ActivationsSelectorWidget()
    w_eval_selector = ActivationsSelectorWidget()
    btn_analyze = widgets.Button(description="Analyze", button_style="primary", disabled=True)
    out_analyze = widgets.Output()
    box_train = widgets.VBox([widgets.HTML("<b>Select activations for analysis</b>"), w_train_selector])
    box_eval = widgets.VBox([widgets.HTML("<b>Select activations for visualization</b>"), w_eval_selector, w_eval_link])
    box_layer2 = widgets.VBox(
        [
            widgets.HTML("<big><b>Analyze activations</b></big>"),
            w_analyze_selector,
            w_readers_selector,
            w_train_separate,
            w_eval_separate,
            widgets.HBox([box_train, box_eval]),
            btn_analyze,
            out_analyze,
        ]
    )

    # Register handlers
    def _update_separate(*args):
        if w_analyze_selector.value == "per layer":
            w_train_separate.description = "Analyze separately per layer"
            w_eval_separate.description = "Visualize separately per layer"
        else:
            w_train_separate.description = "Analyze separately per token"
            w_eval_separate.description = "Visualize separately per token"

    w_analyze_selector.observe(_update_separate, names="value")
    _update_separate()

    def _update_activations(*args):
        CACHE.clear()

        # Update widgets
        activations = w_act_extractor.activations
        w_topk_extractor.set_activations(activations)
        w_train_selector.set_activations(activations)
        w_eval_selector.set_activations(activations)
        btn_analyze.disabled = activations is None
        out_analyze.clear_output()

    w_act_extractor.observe(_update_activations, names="value")
    _update_activations()

    def _update_link(*args):
        if w_eval_link.value:
            w_eval_selector.link(w_train_selector)
        else:
            w_eval_selector.unlink()

    w_eval_link.observe(_update_link, names="value")
    _update_link()

    def _do_plot(*args):
        activations = w_act_extractor.activations
        if activations is None:
            return

        with out_analyze:
            out_analyze.clear_output(wait=True)

            if w_analyze_selector.value == "per layer":
                analyze_fn = analyze_per_layer
                plot_fn = plot_per_layer
            else:
                analyze_fn = analyze_per_token
                plot_fn = plot_per_token

            # Train and cache model
            train_activations = activations.select(
                sample_ids=w_train_selector.samples or None,
                layers=w_train_selector.layers or None,
            )
            readers = CACHE(analyze_fn)(
                train_activations,
                w_readers_selector.readers,
                pool_method=w_train_selector.pool_method,
                exclude_bos=w_train_selector.exclude_bos,
                exclude_special_tokens=w_train_selector.exclude_special_tokens,
                separate=w_train_separate.value,
            )

            # Plot explained variance
            xlabel = "Layer" if w_analyze_selector.value == "per layer" else "Token position"
            plot_reader_statistics(readers, xlabel=xlabel)

            # Plot evaluation
            eval_activations = activations.select(
                sample_ids=w_eval_selector.samples or None,
                layers=w_eval_selector.layers or None,
            )
            plot_fn(
                eval_activations,
                readers,
                pool_method=w_eval_selector.pool_method,
                exclude_bos=w_eval_selector.exclude_bos,
                exclude_special_tokens=w_eval_selector.exclude_special_tokens,
                token_embeddings=w_emb_loader.token_embeddings,
                separate=w_eval_separate.value,
            )

    btn_analyze.on_click(_do_plot)

    # Display widgets
    display(widgets.VBox([box_layer1, widgets.HTML("<hr>"), box_layer2]))


analyze_interactive()

## Manual analysis

### Extract activations

In [ ]:
dataset = load_dataset_from_spec("xstest", max_samples=500)

In [ ]:
model, tokenizer = load_model_and_tokenizer("gemma3_1b")

In [ ]:
activations = extract_activations(model, tokenizer, dataset, apply_chat_template=True)
activations.samples

#### (optional) Save and load activations

In [ ]:
activations.save("../activations/xstest/gemma3_1b/chat")

In [ ]:
activations = Activations.load("../activations/xstest/gemma3_1b/chat")
activations.samples

### Analysis of per-layer trajectories

* Example A: Analyze with PCA the per-layer "cross-sections" of per-token trajectories over last token of all samples
* Example B: Analyze with linear probe and PCA the per-layer trajectories over last 6 tokens of 100 safe and 100 unsafe samples

In [ ]:
readers_per_layer = analyze_per_layer(
    ### Example A ###
    activations,
    ["pca", "pca"],
    pool_method="last",
    separate=True,

    # ### Example B ###
    # activations.select(activations.samples_safe.index[:100].tolist() + activations.samples_unsafe.index[:100].tolist()),
    # ["linear_probe", "pca"],
    # pool_method=slice(-6, None),
    # separate=True,
)
plot_reader_statistics(readers_per_layer, xlabel="Layer")

In [ ]:
plot_per_layer(
    ### Example A ###
    activations,
    readers_per_layer,
    pool_method="last",
    separate=True,

    # ### Example B ###
    # activations.select(activations.samples_safe.index[:100].tolist() + activations.samples_unsafe.index[:100].tolist()),  # train samples
    # # activations.select(activations.samples_safe.index[100:103].tolist() + activations.samples_unsafe.index[100:103].tolist()),  # test samples
    # readers_per_layer,
    # pool_method=slice(-6, None),
    # separate=True,

    ### Optional: Visualize token embeddings ###
    token_embeddings=get_token_embeddings(model, tokenizer, select(range(tokenizer.vocab_size), at_most=0.25)),
)

### Analysis of per-token trajectories

* Example A: Analyze with PCA per-token trajectories over all tokens of the first sample.
* Example B: Analyze with PCA per-token trajectories over last 10 tokens of 3 safe and 3 unsafe samples.

In [ ]:
readers_per_token = analyze_per_token(
    ### Example A ###
    activations.select(activations.samples.index[0]),
    ["pca", "pca"],
    pool_method="all",
    separate=False,

    ### Example B ###
    # activations.select(activations.samples_safe.index[:3].tolist() + activations.samples_unsafe.index[:3].tolist()),
    # ["pca", "pca"],
    # pool_method=slice(-10, None),
    # separate=True,
)
plot_reader_statistics(readers_per_token, xlabel="Token position")

In [ ]:
plot_per_token(
    ### Example A ###
    activations.select(activations.samples.index[0]),
    readers_per_token,
    pool_method="all",
    separate=False,

    ### Example B ###
    # activations.select(activations.samples_safe.index[:3].tolist() + activations.samples_unsafe.index[:3].tolist()),
    # readers_per_token,
    # pool_method=slice(-10, None),
    # separate=True,

    ### Optional: Visualize token embeddings ###
    # token_embeddings=get_token_embeddings(model, tokenizer, select(range(tokenizer.vocab_size), at_most=0.25)),
)

### (optional) Analysis of token embeddings

In [ ]:
token_groups = get_token_groups(tokenizer)
print_token_groups(tokenizer, token_groups)

In [ ]:
plot_token_embeddings(
    model,
    tokenizer,
    token_groups,
    num_components=8,
)

### (optional) Generate responses

In [ ]:
from latent_dynamics.dayang.activations import _prepare_sample


def generate(model, tokenizer, sample, include_response, apply_chat_template):
    sample_prepared = _prepare_sample(
        sample, tokenizer, include_response=include_response, apply_chat_template=apply_chat_template
    )
    inputs = tokenizer(sample_prepared["input"], return_tensors="pt", add_special_tokens=not apply_chat_template)
    outputs = model.generate(**inputs, max_new_tokens=10)
    response = tokenizer.decode(outputs)[0]
    return response


def test_generate(model, tokenizer, sample, include_responses=[False, "Sure", "Sorry"], apply_chat_template=True):
    print(f"ID:             {sample['id']}")
    print(f"is_safe:        {sample['is_safe']}")
    print(f"is_adversarial: {sample['is_adversarial']}")
    for include_response in include_responses:
        print("-" * 50)
        response = generate(
            model,
            tokenizer,
            sample,
            include_response=include_response,
            apply_chat_template=apply_chat_template,
        )
        print(response)


test_generate(model, tokenizer, dataset[0], apply_chat_template=True)